# 00. Dữ liệu, tri thức, đồ thị, đóng băng (P0)

Notebook chuẩn bị mọi thứ **không cần GPU** cho thực nghiệm. Mọi bước **bỏ qua nếu output đã có** trong `pypi_rag_work/` trên Drive.

| Bước | Nội dung | Plan | Thời gian (Colab CPU) |
|---|---|---|---|
| 00 | Thiết lập, test CPU, smoke run với reader `dummy` | - | 3 phút |
| 01 | Clone + pin mal-LLM và PyGuard, kiểm tra license, file cần dùng | mục 2 audit gate | 5-10 phút |
| 02 | Chuẩn hoá 3 bộ dữ liệu, chia tách theo nhóm, kiểm tra rò rỉ | mục 5 | 5-15 phút |
| 03 | Bundle tri thức PyGuard (lexicon API -> hành vi, pattern) | mục 2, mục 10 | 2 phút |
| 04 | Tri thức ngoài: YARA (mẫu trong mal-LLM / YARA Forge), GitHub advisories (tuỳ chọn) | mục 8 | 1-5 phút |
| 05 | KB văn bản chỉ từ **train** + tri thức ngoài, kiểm tra rò rỉ KB | mục 5.3, mục 8 | 5 phút |
| 06 | Chẩn đoán đồ thị hành vi (không LLM): provenance, pattern, path recall | mục 10.5 | 5 phút |
| 07 | Đóng băng: hash sample id theo split, KB, SHA upstream | P0 exit | vài giây |

**Bộ dữ liệu**

| Tên | Nguồn | Nội dung | Vai trò |
|---|---|---|---|
| `malllm` | mal-LLM `RQ_experiments/data/*_packages_final.json` | chỉ `setup.py` + danh sách file; train/test của paper | tái lập baseline, test khoá theo paper |
| `pyguard_obf` | PyGuard `Dataset/PyPI/Experiment/Obfuscation/{Malware,Benign}/*.tgz` | package **đầy đủ** (khoảng 1041 độc, 1028 lành) | bài toán nhiều file, quan hệ hành vi |
| `synthetic` | sinh trong code (`pypi_rag/data/synthetic.py`) | fixture không thực thi được, có nhãn hành vi và đường đi | đo retrieval / path recall / coverage có gold |

**An toàn.** Package nghi độc chỉ được đọc tĩnh (`ast`), trong bộ nhớ, không giải nén ra đĩa, không `pip install`, không import.
Archive gốc nằm trên đĩa máy ảo Colab (`/content/external`), không lên Drive. `pypi_rag_work/data` chứa mã nguồn độc **dạng text**:
đừng đồng bộ thư mục này về máy cá nhân (Drive for desktop), antivirus có thể cách ly file.

## 00. Thiết lập và smoke test
Test CPU và một lần chạy nhỏ trên dữ liệu tổng hợp với reader `dummy` (luật từ khoá, không phải mô hình) để chắc toàn bộ chuỗi code chạy được.

In [ ]:
# ==== 00a. Mount Drive + đường dẫn + helper ====
import os, sys, json, subprocess, time, shutil, glob, random
try:
    from google.colab import drive; drive.mount('/content/drive'); IN_COLAB = True
except ImportError:
    IN_COLAB = False                                        # chạy local để kiểm tra notebook (CPU)

DRIVE_BASE = '/content/drive/MyDrive/ThS/Research/colab' if IN_COLAB else os.environ.get('PYPI_RAG_BASE', os.path.abspath('../..'))  # SỬA NẾU KHÁC
CODE_DIR = f'{DRIVE_BASE}/pypi_malware_rag'                 # folder code này (upload nguyên folder lên Drive)
WORK     = os.environ.get('PYPI_RAG_WORK', f'{DRIVE_BASE}/pypi_rag_work')   # dữ liệu chuẩn hoá, KB, kết quả (giữ qua các phiên)
EXT      = '/content/external' if IN_COLAB else f'{WORK}/external'          # repo upstream + package nghi độc: đĩa máy ảo, KHÔNG để trên Drive
QUICK    = os.environ.get('PYPI_RAG_QUICK') == '1'          # 1 = kiểm tra nhanh: chỉ dữ liệu tổng hợp, reader dummy, không tải gì

os.environ['PYTHONDONTWRITEBYTECODE'] = '1'; os.environ['PYTHONIOENCODING'] = 'utf-8'; sys.dont_write_bytecode = True
for d in glob.glob(f'{CODE_DIR}/**/__pycache__', recursive=True): shutil.rmtree(d, ignore_errors=True)
for d in (WORK, f'{WORK}/data', f'{WORK}/kb', f'{WORK}/results', EXT): os.makedirs(d, exist_ok=True)
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)
assert os.path.exists('pypi_rag/cli.py'), 'chưa upload folder pypi_malware_rag lên Drive (hoặc sai DRIVE_BASE)'
for m in [m for m in sys.modules if m.split('.')[0] == 'pypi_rag']: del sys.modules[m]

CLI = f'{sys.executable} -m pypi_rag.cli'
def sh(cmd, check=True):
    """Chạy lệnh, in output trực tiếp; lỗi -> exception (check=True)."""
    print('$', cmd[:500]); t0 = time.time()
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         encoding='utf-8', errors='replace', cwd=CODE_DIR)
    for line in p.stdout: print(line, end='')
    rc = p.wait(); print(f'exit={rc}  ({(time.time() - t0) / 60:.1f} phút)')
    if check and rc: raise RuntimeError(f'lệnh lỗi (exit {rc}): {cmd[:200]}')
    return rc
def gate(cond, msg): print(('PASS  ' if cond else 'FAIL  ') + msg)
def load(out): return json.load(open(f'{out}/results.json', encoding='utf-8'))
import numpy as np, pandas as pd
pd.set_option('display.max_colwidth', 120); pd.set_option('display.width', 200)
print('code =', CODE_DIR, '| work =', WORK, '| colab =', IN_COLAB, '| quick =', QUICK)

In [ ]:
# ==== 00b. Cài deps + test CPU + smoke (CPU, ~2 phút) ====
if IN_COLAB:
    sh(f'{sys.executable} -m pip install -q "sentence-transformers>=2.2" "transformers>=4.43" accelerate networkx pyyaml')
sh(f'{sys.executable} -m pytest -q -p no:cacheprovider tests/', check=False)
SMOKE = f'{WORK}/smoke'
if not os.path.exists(f'{SMOKE}/run/results.json'):
    sh(f'{CLI} prepare --spec synthetic:120:0 --out {SMOKE}/data')
    sh(f'{CLI} build-kb --data {SMOKE}/data --out {SMOKE}/kb.jsonl')
    sh(f'{CLI} run --data {SMOKE}/data --kb {SMOKE}/kb.jsonl --methods e0 e1 e2 e3 e3a e4 --llm dummy --embedder hash --out {SMOKE}/run --quiet')
print(open(f'{SMOKE}/run/report.md', encoding='utf-8').read()[:1500])
print('\nLưu ý: số của reader "dummy" KHÔNG có ý nghĩa nghiên cứu, chỉ kiểm tra code.')

## 01. Clone và pin upstream (audit gate, plan mục 2)
- **mal-LLM** (MIT): dữ liệu nằm trong Git LFS. Clone không tải LFS rồi chỉ kéo 4 file `*_packages_final.json` (~123 MB). Nếu `git lfs` lỗi thì tải trực tiếp từ `media.githubusercontent.com`.
- **PyGuard**: repo ~3.2 GB và **không có file license**. Chỉ sparse-checkout các thư mục cần đọc lúc chạy; không copy file PyGuard vào repo này, trích dẫn khi dùng.
- Cell kiểm tra các file và hàm mà adapter giả định thực sự tồn tại ở commit đã pin, và in prompt gốc của mal-LLM để đối chiếu với `pypi_rag/generation/prompts.py`.

In [ ]:
# ==== 01. Clone + pin + audit ====
MALLLM_URL,  MALLLM_SHA  = 'https://github.com/malexternalsc/mal-LLM.git', '0491c6c7be7cff45d9d9cd25ca1385d0bf75281b'
PYGUARD_URL, PYGUARD_SHA = 'https://github.com/lxyeternal/PyGuard.git',   'f684d0ab504cea8c0550a1506c2f7d57f827902a'
ONLY = [x for x in os.environ.get('PYPI_RAG_DATASETS', '').split(',') if x]   # chia việc cho nhiều máy, ví dụ 'malllm' hoặc 'pyguard_obf'; rỗng = tất cả
USE_PYGUARD_OBF = not ONLY or 'pyguard_obf' in ONLY   # tải ~660 MB .tgz (package đầy đủ) vào máy ảo Colab
MALLLM, PYGUARD = f'{EXT}/mal-LLM', f'{EXT}/PyGuard'
MALLLM_FILES = ['train_malicious_packages_final.json', 'train_benign_packages_final.json',
                'test_malicious_packages_final.json', 'test_benign_packages_final.json']
audit = {}
if QUICK:
    print('[QUICK] bỏ qua upstream'); MALLLM = PYGUARD = None
else:
    if not os.path.exists(f'{MALLLM}/.git'):
        sh('git lfs install --skip-repo', check=False)
        sh(f'GIT_LFS_SKIP_SMUDGE=1 git clone -q {MALLLM_URL} {MALLLM}')
    sh(f'cd {MALLLM} && GIT_LFS_SKIP_SMUDGE=1 git checkout -q {MALLLM_SHA}')
    sh(f'cd {MALLLM} && git lfs pull --include="RQ_experiments/data/*_packages_final.json"', check=False)
    import urllib.request
    for fn in MALLLM_FILES:
        p = f'{MALLLM}/RQ_experiments/data/{fn}'
        if open(p, 'rb').read(40).startswith(b'version https://git-lfs'):
            url = f'https://media.githubusercontent.com/media/malexternalsc/mal-LLM/{MALLLM_SHA}/RQ_experiments/data/{fn}'
            print('tải trực tiếp', url); urllib.request.urlretrieve(url, p)
        print(f'  {fn}: {os.path.getsize(p) / 1e6:.1f} MB')
    SPARSE = ['Resources/Taxonomy', 'Core/ActionSequence/action_sequences',
              'Core/PatternGenerator/analysis_results/optimized_patterns', 'Core/PatternGenerator/pattern_results',
              'Core/PatternGenerator/hierarchical_patterns'] + (['Dataset/PyPI/Experiment/Obfuscation'] if USE_PYGUARD_OBF else [])
    if not os.path.exists(f'{PYGUARD}/.git'):
        sh(f'git clone -q --filter=blob:none --no-checkout {PYGUARD_URL} {PYGUARD}')
    sh(f'cd {PYGUARD} && git sparse-checkout init --cone && git sparse-checkout set ' + ' '.join(SPARSE) + f' && git checkout -q {PYGUARD_SHA}')
    head = lambda r: subprocess.check_output(['git', '-C', r, 'rev-parse', 'HEAD'], text=True).strip()
    lic = lambda r: [os.path.basename(x) for x in glob.glob(f'{r}/LICEN[CS]E*') + glob.glob(f'{r}/COPYING*')]
    audit = {'mal-LLM': {'url': MALLLM_URL, 'sha': head(MALLLM), 'license_files': lic(MALLLM)},
             'PyGuard': {'url': PYGUARD_URL, 'sha': head(PYGUARD), 'license_files': lic(PYGUARD),
                         'note': 'không có license: chỉ đọc lúc chạy (sparse checkout), không phân phối lại, trích dẫn'}}
    gate(audit['mal-LLM']['sha'] == MALLLM_SHA, f"mal-LLM pin {MALLLM_SHA[:10]}, license {audit['mal-LLM']['license_files']}")
    gate(audit['PyGuard']['sha'] == PYGUARD_SHA, f"PyGuard pin {PYGUARD_SHA[:10]}, license {audit['PyGuard']['license_files'] or 'KHÔNG CÓ'}")
    EXPECT = [f'{MALLLM}/RQ_experiments/data/{f}' for f in MALLLM_FILES] + [
        f'{MALLLM}/RQ_experiments/Zero_shot_baseline_llama3_1_8b/simulate_test.py',
        f'{MALLLM}/RQ_experiments/RAG_experiments/Simple_RAG/simulate_yara_rag.py',
        f'{MALLLM}/RQ_experiments/YaraForge/sample-yara-rules-core.yar',
        f'{PYGUARD}/Resources/Taxonomy/action_categories.json',
        f'{PYGUARD}/Core/ActionSequence/action_sequences',
        f'{PYGUARD}/Core/PatternGenerator/analysis_results/optimized_patterns/optimized_patterns_id.json'] + (
        [f'{PYGUARD}/Dataset/PyPI/Experiment/Obfuscation/Malware', f'{PYGUARD}/Dataset/PyPI/Experiment/Obfuscation/Benign'] if USE_PYGUARD_OBF else [])
    for p in EXPECT: gate(os.path.exists(p), 'tồn tại: ' + p.replace(EXT + '/', ''))
    audit['expected_files_present'] = all(os.path.exists(p) for p in EXPECT)
    src = open(f'{MALLLM}/RQ_experiments/Zero_shot_baseline_llama3_1_8b/simulate_test.py', encoding='utf-8', errors='replace').read()
    i = src.find('cybersecurity expert'); print('\n--- prompt gốc mal-LLM (đối chiếu prompts.MALLLM_DEV) ---\n', src[max(0, i - 40): i + 900])
    json.dump(audit, open(f'{WORK}/upstream_audit.json', 'w'), indent=2)

## 02. Chuẩn hoá, chia tách theo nhóm, kiểm tra rò rỉ (plan mục 5)
- Nhóm = cùng tên package (chuẩn hoá) hoặc cùng family hoặc cùng hash archive hoặc near-duplicate (MinHash Jaccard >= 0.9). Một nhóm không bao giờ nằm ở hai split.
- `malllm`: giữ **test khoá** của paper (`--strategy upstream`), tách validation từ train theo nhóm. Mẫu test trùng nhóm với train được đánh dấu `cross_split_dup`; notebook 01 báo cáo cả bản đầy đủ (so được với paper) và bản bỏ trùng.
- `pyguard_obf`, `synthetic`: 70/15/15 sau khi gộp nhóm, phân tầng theo nhãn.
- Nhãn chỉ có `benign` / `malicious` / `unknown`; `unknown` không tính điểm. Không có nhãn đoán tự động.

In [ ]:
# ==== 02. prepare ====
DATASETS = {'malllm': dict(spec=f'malllm:{MALLLM}', strategy='upstream'),
            'pyguard_obf': dict(spec=f'pyguard_obf:{PYGUARD}', strategy='grouped'),
            'synthetic': dict(spec='synthetic:300:0', strategy='grouped'),
            'cases': dict(spec='cases:', strategy='keep')}          # C01-C06 của tài liệu test case, toàn bộ là test
if QUICK: DATASETS = {'synthetic': dict(spec='synthetic:200:0', strategy='grouped'), 'cases': dict(spec='cases:', strategy='keep')}
elif not USE_PYGUARD_OBF: DATASETS.pop('pyguard_obf')
if ONLY and not QUICK: DATASETS = {k: v for k, v in DATASETS.items() if k in ONLY or k in ('synthetic', 'cases')}
STATS = {}
for name, d in DATASETS.items():
    out = f'{WORK}/data/{name}'
    if not os.path.exists(f'{out}/stats.json'):
        sh(f"{CLI} prepare --config configs/dataset.yaml --spec {d['spec']} --strategy {d['strategy']} --out {out}")
    STATS[name] = json.load(open(f'{out}/stats.json'))
display(pd.DataFrame({n: s['split_report']['counts'] for n, s in STATS.items()}).fillna(0).astype(int))
display(pd.DataFrame({n: {k: v for k, v in s['leakage'].items() if k != 'counts'} for n, s in STATS.items()}))
for n, s in STATS.items():
    L, up = s['leakage'], s['split_report']['strategy'] == 'upstream'
    print(f"--- {n}: {s['n']} mẫu, {s['no_source']} mẫu không có file nguồn, nhóm lớn nhất {s['split_report']['largest_group']}")
    if up:
        print(f"    test trùng nhóm với train (split của paper): {s['split_report'].get('test_cross_split_dup', 0)} mẫu -> báo cáo cả hai bản")
    else:
        gate(L['package_name_across_splits'] == 0 and L['family_across_splits'] == 0, f'{n}: không tên package / family nào nằm ở hai split')
        gate(L['near_dup_pairs_across_splits'] == 0, f'{n}: không cặp near-duplicate nào vắt qua hai split')
        gate(L['exact_archive_dup_across_splits'] == 0, f'{n}: không archive trùng hash giữa các split')

## 03. Tri thức hành vi PyGuard
`pyguard-bundle` đọc taxonomy (`action_categories.json`), các ánh xạ `api_name -> action id` do LLM của PyGuard sinh (`action_sequences/*`),
và pattern khai phá (`optimized_patterns_id.json`). Kết quả là một file JSON nhỏ trên Drive nên các phiên sau không cần checkout lại.
Mặc định, PyGuard vào hệ thống ở hai chỗ: tài liệu pattern cho E1/E2/E3a và node `Pattern` (khớp chuỗi action) cho E3.
Lexicon **không** thay bộ phân tích tĩnh dùng chung, trừ khi bật `--pyguard-taxonomy` (ablation), để E0 không nhận tri thức ngoài.
File `.pkl` của PyGuard không bao giờ được nạp: unpickle file không tin cậy có thể chạy code.

In [ ]:
# ==== 03. PyGuard bundle ====
BUNDLE = f'{WORK}/kb/pyguard_bundle.json'
if PYGUARD and not os.path.exists(BUNDLE):
    sh(f'{CLI} pyguard-bundle --repo {PYGUARD} --out {BUNDLE}')
if os.path.exists(BUNDLE):
    B = json.load(open(BUNDLE))
    lex = pd.DataFrame(B['lexicon']).T.sort_values('n', ascending=False)
    print(B['origin'], '| API trong lexicon:', len(lex), '| có hành vi:', int((lex['behaviors'].str.len() > 0).sum()))
    display(lex.head(15)[['action_id', 'n', 'behaviors']])
    print('pattern theo loại:', pd.Series([p['type'] for p in B['patterns']]).value_counts().to_dict())
    gate(len(B['patterns']) >= 100, f"đọc được {len(B['patterns'])} pattern PyGuard")
else:
    BUNDLE = None; print('(không có bundle PyGuard)')

## 04. Tri thức ngoài cho Simple RAG kiểu mal-LLM
mal-LLM index (1) luật YARA do gpt-4o-mini diễn giải, (2) GitHub advisories, (3) setup.py độc trong train. Ở đây: luật YARA được index nguyên văn
(tên, meta, strings, condition), không viết lại bằng LLM. Repo mal-LLM chỉ có **bản mẫu** 7 KB; bật `YARA_FULL` để tải YARA Forge core.
Advisories nêu tên package, nên bước 05 loại mọi advisory nhắc tới package của val/test.

In [ ]:
# ==== 04. YARA + advisories ====
YARA_FULL = False          # True: tải YARA Forge core (bộ mal-LLM dùng ngoài repo)
FETCH_ADVISORIES = False   # True: cần mạng; nên thêm secret GITHUB_TOKEN trong Colab
YARA = f'{MALLLM}/RQ_experiments/YaraForge/sample-yara-rules-core.yar' if MALLLM else None
if YARA_FULL and not QUICK:
    import urllib.request, zipfile
    z = f'{WORK}/kb/yara-forge-rules-core.zip'
    try:
        if not os.path.exists(z):
            urllib.request.urlretrieve('https://github.com/YARAHQ/yara-forge/releases/latest/download/yara-forge-rules-core.zip', z)
        zipfile.ZipFile(z).extractall(f'{WORK}/kb/yara_forge'); YARA = f'{WORK}/kb/yara_forge'
    except Exception as e:
        print('không tải được YARA Forge, dùng bản mẫu:', e)
ADV = f'{WORK}/kb/advisories_pip.json'
if FETCH_ADVISORIES and not QUICK and not os.path.exists(ADV):
    try:
        from google.colab import userdata; os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN') or ''
    except Exception: pass
    from pypi_rag.knowledge.external import fetch_advisories
    print('advisories:', fetch_advisories(ADV))
ADV = ADV if os.path.exists(ADV) else None
if YARA:
    from pypi_rag.knowledge.external import load_yara
    ys = load_yara(YARA); print(len(ys), 'luật YARA, ví dụ:\n', ys[0].render(500) if ys else '-')

## 05. KB văn bản và kiểm tra rò rỉ KB
Nguồn theo bộ dữ liệu: luật/pattern tự viết (`builtin`), ví dụ mã từ **train** (tên package bị che), setup.py độc trong train (`malllm_code`, index thứ ba của mal-LLM),
pattern PyGuard, YARA, advisories. Kiểm tra rò rỉ loại: tài liệu sinh từ mẫu val/test, tài liệu chứa nguyên văn file của val/test, advisory nhắc tên package val/test.

In [ ]:
# ==== 05. build-kb ====
def kb_sources(name):
    if name == 'cases': return ['builtin', 'case_rules']   # tri thức của tài liệu test case, cố định trước khi chạy
    s = ['builtin', 'examples']
    if name == 'malllm': s.append('malllm_code')
    if BUNDLE: s.append('pyguard')
    if YARA and name != 'synthetic': s.append('yara')
    if ADV and name != 'synthetic': s.append('advisories')
    return s
for name in DATASETS:
    kb = f'{WORK}/kb/{name}.jsonl'
    if not os.path.exists(kb):
        sh(f"{CLI} build-kb --config configs/retrieval.yaml --data {WORK}/data/{name} --out {kb} --sources {' '.join(kb_sources(name))}"
           + (f' --pyguard-bundle {BUNDLE}' if BUNDLE else '') + (f' --yara {YARA}' if YARA else '') + (f' --advisories {ADV}' if ADV else ''))
    rep = json.load(open(kb + '.report.json'))
    print(f"--- {name}: {rep['n_docs']} tài liệu {rep['by_source_type']} | bỏ do rò rỉ: {rep['n_dropped']}")
    gate(rep['leakage_after']['clean'], f'{name}: KB sạch (không tài liệu từ val/test, không mã val/test nguyên văn, không advisory nêu tên package val/test)')

## 06. Chẩn đoán đồ thị hành vi (không LLM, plan mục 10.5)
Mỗi cạnh phải có `source_file, line_start, line_end, extractor, confidence, evidence_status`. `FLOWS_TO` chỉ tạo khi có chuỗi def-use / lồng gọi trong AST.
Trên dữ liệu tổng hợp có `gold_paths` nên đo được **path recall**. Mẫu `m_py2_exfil` (Python 2, không parse được) là ca thiếu cạnh có chủ đích.

In [ ]:
# ==== 06. build-graph ====
from pypi_rag.schema import read_jsonl
from pypi_rag.evaluation.evidence import path_recall
MAX_GRAPH = 1000
GSUM = {}
for name in DATASETS:
    out = f'{WORK}/graphs/{name}'
    if not os.path.exists(f'{out}/graphs_test.summary.json'):
        sh(f'{CLI} build-graph --data {WORK}/data/{name} --split test --out {out} --max-items {MAX_GRAPH}' + (f' --pyguard-bundle {BUNDLE}' if BUNDLE else ''))
    GSUM[name] = json.load(open(f'{out}/graphs_test.summary.json'))
display(pd.DataFrame(GSUM).T.round(3))
for name, g in GSUM.items():
    gate(g['missing_provenance_edges'] == 0, f'{name}: mọi cạnh có đủ provenance')
    gate(g['pattern_hit_rate_malicious'] > g['pattern_hit_rate_benign'],
         f"{name}: pattern khớp ở mẫu độc nhiều hơn mẫu lành ({g['pattern_hit_rate_malicious']:.2f} vs {g['pattern_hit_rate_benign']:.2f})")
rows = [r for r in read_jsonl(f'{WORK}/graphs/synthetic/graphs_test.jsonl') if r['gold_paths']]
for name, sel in (('parse được', [r for r in rows if r['parse_coverage'] >= 1]), ('không parse được (Python 2)', [r for r in rows if r['parse_coverage'] < 1])):
    pr = [path_recall([p['apis'] or [] for p in r['paths']], r['gold_paths']) for r in sel]
    print(f'path recall synthetic/test, {name}: {np.mean(pr) if pr else float("nan"):.3f} ({len(pr)} mẫu)')
    if name == 'parse được' and pr: gate(np.mean(pr) >= 0.9, 'path recall >= 0.9 trên file parse được')
print('File không parse được chỉ có thao tác tìm bằng regex, không có cạnh FLOWS_TO: đây là ca "thiếu cạnh" có chủ đích (plan mục 10.5).')

## 07. Đóng băng (exit criterion P0: cùng sample id và hash ở mọi lần chạy)

In [ ]:
# ==== 07. FROZEN.json ====
import hashlib
def file_sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()[:16]
frozen = {'upstream': audit or (json.load(open(f'{WORK}/upstream_audit.json')) if os.path.exists(f'{WORK}/upstream_audit.json') else {}),
          'datasets': {}, 'kb': {}, 'pyguard_bundle': file_sha(BUNDLE) if BUNDLE else None}
for name in DATASETS:
    by = {}
    for r in read_jsonl(f'{WORK}/data/{name}/manifest.jsonl'):
        by.setdefault(r['split'], []).append(f"{r['sample_id']}:{r['sha256']}:{r['label']}")
    frozen['datasets'][name] = {sp: {'n': len(v), 'sha': hashlib.sha256('\n'.join(sorted(v)).encode()).hexdigest()[:16]} for sp, v in by.items()}
    frozen['kb'][name] = file_sha(f'{WORK}/kb/{name}.jsonl')
F = f'{WORK}/FROZEN.json'
if os.path.exists(F):
    old = json.load(open(F))
    gate(old['datasets'] == frozen['datasets'], 'sample id + hash + nhãn của mọi split trùng lần đóng băng trước')
    gate(old['kb'] == frozen['kb'], 'KB không đổi so với lần đóng băng trước')
else:
    json.dump(frozen, open(F, 'w'), indent=2); print('đã ghi', F)
display(pd.DataFrame({n: {sp: v['n'] for sp, v in d.items()} for n, d in frozen['datasets'].items()}).fillna(0).astype(int))